In [145]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


In [32]:
x = torch.rand((4,6))
linear = nn.Linear(6, 10)
x

tensor([[0.0530, 0.0499, 0.4677, 0.8757, 0.5561, 0.7984],
        [0.9758, 0.2482, 0.1469, 0.4345, 0.6988, 0.8883],
        [0.2638, 0.2658, 0.1375, 0.4610, 0.7439, 0.0351],
        [0.1422, 0.4056, 0.5341, 0.5862, 0.1469, 0.2960]])

In [ ]:
# GLOBALS 
token_amount = 50
B,T,C  = 4, 8 ,32 # Batch size, time steps (seq length), channels (embedding vector size per token)
seq_len = 8


In [34]:
x = linear(x)
x

tensor([[-0.2551, -0.2710, -0.5838, -0.3300, -0.2921,  0.2797,  0.1288, -0.0140,
          0.1808, -0.8177],
        [-0.4697, -0.0342, -0.5842, -0.2817, -0.1733,  0.4389,  0.3472, -0.0368,
          0.4078, -0.4071],
        [-0.1606, -0.1914, -0.3775, -0.2417,  0.0681,  0.2261,  0.1644, -0.1374,
          0.4444, -0.6783],
        [ 0.1483, -0.3018, -0.4926, -0.2763, -0.1729,  0.2476,  0.0311, -0.0176,
          0.3249, -0.7794]], grad_fn=<AddmmBackward0>)

In [35]:
ten = 2 * torch.rand((5,5)) - 1
ten

tensor([[-0.9606, -0.9881, -0.4695,  0.7320,  0.2984],
        [-0.2569, -0.8423,  0.6759, -0.7158, -0.4976],
        [ 0.7535, -0.4182,  0.6187,  0.6577,  0.1092],
        [ 0.0532, -0.2014, -0.1886,  0.7889,  0.4521],
        [ 0.3777,  0.6784,  0.7436,  0.0491,  0.8127]])

In [36]:
approx = 'None' # 'tanh'
gelu = nn.GELU()
y = gelu(ten)
y

tensor([[-0.1617, -0.1596, -0.1499,  0.5621,  0.1842],
        [-0.1024, -0.1683,  0.5073, -0.1697, -0.1539],
        [ 0.5835, -0.1413,  0.4529,  0.4898,  0.0593],
        [ 0.0277, -0.0846, -0.0802,  0.6192,  0.3049],
        [ 0.2444,  0.5097,  0.5736,  0.0255,  0.6435]])

In [37]:
relu = nn.ReLU()
y_rel = relu(ten)
y_rel

tensor([[0.0000, 0.0000, 0.0000, 0.7320, 0.2984],
        [0.0000, 0.0000, 0.6759, 0.0000, 0.0000],
        [0.7535, 0.0000, 0.6187, 0.6577, 0.1092],
        [0.0532, 0.0000, 0.0000, 0.7889, 0.4521],
        [0.3777, 0.6784, 0.7436, 0.0491, 0.8127]])

In [85]:
torch.rand((3,6))

tensor([[0.8548, 0.6312, 0.9565, 0.6963, 0.7304, 0.0062],
        [0.0961, 0.4339, 0.2332, 0.1106, 0.8382, 0.8258],
        [0.3358, 0.6122, 0.2340, 0.2256, 0.5206, 0.6772]])

In [75]:
B,T,C  = 4, 8 ,32
x = torch.rand(B,T,C)

In [40]:
x.shape

torch.Size([4, 8, 32])

In [41]:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

In [42]:
k = key(x)
q = query(x)
k.shape, q.shape

(torch.Size([4, 8, 16]), torch.Size([4, 8, 16]))

In [43]:
wei = q @ k.transpose(-1, -2)
tril = torch.tril(torch.ones(T, T))
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [44]:
wei  = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[[ 0.0483,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 0.3152,  0.4347,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 0.4779,  0.3227,  0.1709,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 0.2862,  0.0790, -0.0807, -0.0122,    -inf,    -inf,    -inf,
             -inf],
         [ 0.1111, -0.2673, -0.3838, -0.0689, -0.4181,    -inf,    -inf,
             -inf],
         [ 0.4140,  0.2287,  0.0250,  0.3047,  0.1456,  0.5605,    -inf,
             -inf],
         [ 0.3361,  0.2113,  0.1471,  0.2418,  0.1907,  0.5282,  0.3753,
             -inf],
         [ 0.4636,  0.2061,  0.4341,  0.2645,  0.1823,  0.6890,  0.3837,
           0.0960]],

        [[-0.2214,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [-0.3037, -0.0860,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [-0.1672,  0.4552,  0.3228,    -inf,    -inf,    -inf,    -

In [45]:
wei = F.softmax(wei, dim=1)
wei

tensor([[[0.0955, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1248, 0.1818, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1468, 0.1626, 0.1820, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1212, 0.1274, 0.1415, 0.1688, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1017, 0.0901, 0.1045, 0.1595, 0.1558, 0.0000, 0.0000, 0.0000],
         [0.1377, 0.1480, 0.1573, 0.2317, 0.2738, 0.3220, 0.0000, 0.0000],
         [0.1274, 0.1454, 0.1778, 0.2176, 0.2864, 0.3118, 0.4979, 0.0000],
         [0.1447, 0.1447, 0.2368, 0.2226, 0.2840, 0.3662, 0.5021, 1.0000]],

        [[0.1181, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1087, 0.0819, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1246, 0.1407, 0.1342, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1536, 0.1732, 0.2256, 0.2005, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1374, 0.1574, 0.1611, 0.2131, 0.2797, 0.0000, 0.0000, 0.0000],
         [0.1138, 0.098

In [46]:
wei.shape

torch.Size([4, 8, 8])

In [47]:
v = value(x)

In [48]:
v.shape

torch.Size([4, 8, 16])

In [49]:
att = wei @ v

In [50]:
att.shape

torch.Size([4, 8, 16])

In [51]:
test = torch.ones(5)

In [52]:
test[0:2] = test[:2] + 3

In [53]:
test.std()

tensor(1.6432)

In [54]:
x.shape

torch.Size([4, 8, 32])

In [55]:
emb = nn.Embedding(token_amount, C)

In [92]:
torch.manual_seed(47)
input_sequences = torch.randint(0, token_amount, (B, T))
input_sequences

tensor([[ 5, 48, 49,  4,  0, 21, 36,  1],
        [23,  2, 41, 25,  3, 49, 17,  8],
        [22, 40, 19, 11, 27, 21, 38, 42],
        [45, 28, 49, 49, 47, 20, 35, 40]])

In [93]:
# Here we're doing the indexing. For each sequence example, fetch the corresponding embedding matrix.
# `input_sequences` here should be B examples of T token sequences in which each T element is a token ID not some rand value
input_embedding = emb(input_sequences)
input_embedding.shape # B T C

torch.Size([4, 8, 32])

In [ ]:
pos_embedding = nn.Embedding(seq_len, C) # For each position, we have an embedding vector.
pos = torch.arange(0, T)
positional_embeddings = pos_embedding(pos)
positional_embeddings.shape

torch.Size([8, 32])

In [104]:
positional_embeddings[0]

tensor([ 1.3891,  0.7400, -0.2291,  1.6027, -1.6598,  0.6356, -0.0039,  1.3053,
         0.2978, -1.3682, -0.1116,  1.3974,  0.1719,  1.1234,  1.2797, -0.6540,
        -1.3221, -0.9979, -0.3226, -1.1057,  0.0352,  1.0379,  0.3378,  0.7722,
        -1.4025,  0.1434,  2.1679, -0.0653,  0.1861, -0.4850, -0.2158, -1.0045],
       grad_fn=<SelectBackward0>)

In [109]:
input_embedding[0][0]

tensor([-1.7554,  0.6182,  0.9372, -0.3557,  1.1316, -0.7013, -0.6912,  1.4648,
        -1.0783,  0.7145,  0.6758, -0.6752,  0.6620,  0.3093,  1.0519, -0.7792,
        -0.7271,  0.8223, -0.3805, -0.5830,  2.6497, -1.4848,  0.8006,  0.8374,
        -2.4803,  1.1982,  0.3057, -0.4137, -1.0701,  0.5630,  0.5534, -1.0252],
       grad_fn=<SelectBackward0>)

In [120]:
x = input_embedding +  positional_embeddings
f'{x[0][0][0].item():4f}, {(1.3891 + -1.7554):4f}' # == (1.3891 + -1.7554)

'-0.366344, -0.366300'

In [138]:
test_tensor = torch.arange(0, 20).view(5, 4)
test_tensor

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]])

In [140]:
test_tensor.split(2, 1)

(tensor([[ 0,  1],
         [ 4,  5],
         [ 8,  9],
         [12, 13],
         [16, 17]]),
 tensor([[ 2,  3],
         [ 6,  7],
         [10, 11],
         [14, 15],
         [18, 19]]))

In [144]:
tril = torch.tril(torch.ones(T,T))
mask = tril.masked_fill(tril == 0, float('-inf'))
mask

tensor([[1., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., 1., -inf, -inf, -inf],
        [1., 1., 1., 1., 1., 1., -inf, -inf],
        [1., 1., 1., 1., 1., 1., 1., -inf],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [147]:
@dataclass
class GPTConfig:
    seq_len: int = 8
    embedding_len = 32
    n_heads = 8
    n_blocks = 0

class SelfAttention(nn.Module):
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.n_heads = config.n_heads
        self.embedding_len = config.embedding_len
        # Each atten head processes an equal portion of the embedding so the numbers must be compatible.
        assert config.embedding_len % config.n_heads == 0
        self.qkv_lin = nn.Linear(config.embedding_len, config.embedding_len * 3)
        self.proj_lin = nn.Linear(config.embedding_len, config.embedding_len)

        # Add stencil here? better with register buffer so it's no grad

    def forward(self, x):
        # rough steps: get QKV > matmul q with k > get mask > apply mask > softmax 
        # > matmul softmax result (wei) with V -> project? (or is that done before?) 
        B, T, C = x.size() 
        # XXX: Would C being smaller than embed_len cause compatibility issue with n_heads?
        # NO!!! C disappears after linear matmul. But wouldn't it prevent matmul cus of mismatch? (B,C) @ (n_emb, n_emb) if C != n_emb > crash
        assert C <= self.config.embedding_len, "wtf?" 
        qkv = self.qkv_lin(x) # B T EMBED_LEN*3
        q, k, v = qkv.split(self.config.embedding_len, -1) # split over emb dim
        
        # Pre process for multihead attn. 
        # For each example, for each head, for each timestep we have headsize result
        q = q.view(B, T, self.n_heads, self.n_heads // self.embedding_len).transpose(1,2) # (B, n_heads, T, head_size) 
        k = k.view(B, T, self.n_heads, self.n_heads // self.embedding_len).transpose(1,2)
        v = v.view(B, T, self.n_heads, self.n_heads // self.embedding_len).transpose(1,2)
        # (B, n_heads, T, head_size) @ (B, n_heads, head_size, T) = (B, n_heads, T, T)?
        wei = q @ k.transpose(-1, -2)
        with torch.no_grad():
            tril = torch.tril(torch.ones(T,T))
            wei = wei.masked_fill(tril == 0, float('-inf'))
            print(wei)

In [148]:
s_attn = SelfAttention(GPTConfig())

In [158]:
x = torch.rand(4, GPTConfig.seq_len, GPTConfig.embedding_len)
x.shape

torch.Size([4, 8, 32])

In [160]:
out = s_attn(x)

RuntimeError: shape '[4, 8, 8, 0]' is invalid for input of size 1024